In [1]:
import torch as t
import numpy as np
import sys, platform, multiprocessing, os, random, tqdm

from pathlib import Path
from captum.attr import FeatureAblation

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "apps" / "search_agent").is_dir())
APPS_ROOT = REPO_ROOT / "apps"
if str(APPS_ROOT) not in sys.path:
    sys.path.insert(0, str(APPS_ROOT))

APP_ROOT = APPS_ROOT / "search_agent"

from search_agent.search.deep_learn.cost_model import CostApproximation, EMBEDDING_DIMENSION, NUM_SELECTABLE_LEXICAL_KEYS, NUM_WINK_POS_TAGS
from search_agent.search.deep_learn.cost_apprx import MapCostDataset
from search_agent.search.deep_learn.dataset import make_index_batch_sampler

In [2]:
USE_MULTIPROCESSING = False

if platform.system() in ["Darwin", "Linux"] and USE_MULTIPROCESSING:
    multiprocessing.set_start_method("fork", force=True)

MP_CONTEXT = multiprocessing.get_start_method() if USE_MULTIPROCESSING else None
NUM_WORKERS = 4 if USE_MULTIPROCESSING else 0
USE_PERSISTENT_WORKERS = False

DEVICE = t.device("cuda" if t.cuda.is_available() else "mps" if t.backends.mps.is_available() else "cpu")
CUDA = DEVICE.type == "cuda"
PIN_MEMORY = CUDA
NON_BLOCKING = PIN_MEMORY

print(f"Using device={DEVICE}")
print(f"Using multiprocessing context={MP_CONTEXT}, workers={NUM_WORKERS}")


def set_seed(seed):
    """Seed Python, NumPy, and PyTorch for reproducible notebook runs."""
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["PYTHONHASHSEED"] = str(seed)

    t.manual_seed(seed)
    t.cuda.manual_seed(seed)
    t.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    t.backends.cudnn.deterministic = True
    t.backends.cudnn.benchmark = False
    t.use_deterministic_algorithms(True)


MASTER_SEED = 42
set_seed(MASTER_SEED)
G = t.Generator().manual_seed(MASTER_SEED)


def worker_init_fn(worker_id):
    """Seed each DataLoader worker from the notebook seed."""
    worker_seed = MASTER_SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    t.manual_seed(worker_seed)


DATALOADER_WORKER_INIT_FN = worker_init_fn if NUM_WORKERS else None
DATALOADER_PERSISTENT_WORKERS = USE_PERSISTENT_WORKERS if NUM_WORKERS else False
DATALOADER_MP_CONTEXT = MP_CONTEXT if NUM_WORKERS else None


Using device=cuda
Using multiprocessing context=None, workers=0


In [4]:
set_seed(MASTER_SEED)
G.manual_seed(MASTER_SEED)

# Define the sizes based on your code
EMB_DIM = EMBEDDING_DIMENSION
COST_FEATURE_COUNT = EMB_DIM * 4 + 3 + 1 + NUM_SELECTABLE_LEXICAL_KEYS + NUM_WINK_POS_TAGS

# Initialize mask
feature_mask = t.zeros((COST_FEATURE_COUNT), dtype=t.long)
offset = 0
group_id = 0

# 1. Raw Endpoints
feature_mask[offset : offset + EMB_DIM] = group_id # current_embedding
offset += EMB_DIM; group_id += 1

feature_mask[offset : offset + EMB_DIM] = group_id # target_embedding
offset += EMB_DIM; group_id += 1

# 2. Pair Interactions (Vector)
feature_mask[offset : offset + EMB_DIM] = group_id # difference vector
offset += EMB_DIM; group_id += 1

feature_mask[offset : offset + EMB_DIM] = group_id # product vector
offset += EMB_DIM; group_id += 1

# 3. Pair Interactions (Scalar)
feature_mask[offset] = group_id # cosine similarity
offset += 1; group_id += 1

feature_mask[offset] = group_id # euclidean distance
offset += 1; group_id += 1

feature_mask[offset] = group_id # dot product
offset += 1; group_id += 1

# 4. Graph & Search Constraints
feature_mask[offset] = group_id # lemmatized
offset += 1; group_id += 1

feature_mask[offset : offset + NUM_SELECTABLE_LEXICAL_KEYS] = group_id # lexical_field_mask
offset += NUM_SELECTABLE_LEXICAL_KEYS; group_id += 1

feature_mask[offset:] = group_id # pos_mask

group_names = [
    "current_embedding", "target_embedding", 
    "difference_vector", "product_vector", 
    "cosine_similarity", "euclidean_distance", "dot_product",
    "lemmatized", "lexical_field_mask", "pos_mask"
]

In [6]:
set_seed(MASTER_SEED)
G.manual_seed(MASTER_SEED)

model = CostApproximation.load_model(
    APP_ROOT / "search/deep_learn/models/best_CostApproximation.pt", DEVICE
)

def model_forward(x):
    model_output = model.forward(x)
    return t.stack([model_output.cost, model_output.reachable_logit], dim=1)  # Stack with zeros to create a two-class output



ablator = FeatureAblation(model_forward)

In [8]:
set_seed(MASTER_SEED)
G.manual_seed(MASTER_SEED)

dataset_path = APP_ROOT / "search/deep_learn/data/cost_approx_random_sampling_dataset_1.pt"
dataset = MapCostDataset.load(dataset_path)

In [36]:
set_seed(MASTER_SEED)
G.manual_seed(MASTER_SEED)

dataloader = t.utils.data.DataLoader(
    dataset,
    batch_size=512,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    worker_init_fn=DATALOADER_WORKER_INIT_FN,
    persistent_workers=DATALOADER_PERSISTENT_WORKERS,
    multiprocessing_context=DATALOADER_MP_CONTEXT,
)

In [37]:
set_seed(MASTER_SEED)
G.manual_seed(MASTER_SEED)

model.eval()

n_instances = len(dataloader)

cost_attrs = t.zeros((n_instances, COST_FEATURE_COUNT), dtype=t.float32)
reachability_attrs = t.zeros((n_instances, COST_FEATURE_COUNT), dtype=t.float32)

for features, labels in tqdm.tqdm(dataloader):
    features = features.to(DEVICE, non_blocking=NON_BLOCKING)
    cost_attributions = ablator.attribute(
        features,
        feature_mask=feature_mask,
        target=0,  
        perturbations_per_eval=10
    )
    
    cost_attrs[len(cost_attrs) - len(features):len(cost_attrs)] = cost_attributions.abs()
    
    reachability_attributions = ablator.attribute(
        features,
        feature_mask=feature_mask,
        target=1,  
        perturbations_per_eval=10
    )
    reachability_attrs[len(reachability_attrs) - len(features):len(reachability_attrs)] = reachability_attributions.abs()




  0%|          | 0/977 [00:00<?, ?it/s]

100%|██████████| 977/977 [00:41<00:00, 23.41it/s]


In [38]:
mean_cost_attr = cost_attrs.mean(dim=0)
std_cost_attr = cost_attrs.std(dim=0)
normalised_cost_attr = (mean_cost_attr - mean_cost_attr.min()) / (
    mean_cost_attr.max() - mean_cost_attr.min()
)

mean_reachability_attr = reachability_attrs.mean(dim=0)
std_reachability_attr = reachability_attrs.std(dim=0)
normalised_reachability_attr = (mean_reachability_attr - mean_reachability_attr.min()) / (
    mean_reachability_attr.max() - mean_reachability_attr.min()
)

total_normalised_attr = normalised_cost_attr + normalised_reachability_attr

print("Feature Attributions (Mean ± Std, Normalised):")
for i, name in enumerate(group_names):
    
    start_idx = (feature_mask == i).nonzero(as_tuple=True)[0][0]
    end_idx = (feature_mask == i).nonzero(as_tuple=True)[0][-1] + 1
    # print((feature_mask == i).nonzero(as_tuple=True)[0][0])

    cost_score = mean_cost_attr[start_idx:end_idx].mean().item()
    cost_std = std_cost_attr[start_idx:end_idx].mean().item()
    reach_score = mean_reachability_attr[start_idx:end_idx].mean().item()
    reach_std = std_reachability_attr[start_idx:end_idx].mean().item()
    total_score = total_normalised_attr[start_idx:end_idx].mean().item()

    print(
        f"{name}: Cost = {cost_score:.4f} ± {cost_std:.4f}, Reachability = {reach_score:.4f} ± {reach_std:.4f}, Total Normalised = {total_score:.4f}"
    )

Feature Attributions (Mean ± Std, Normalised):
current_embedding: Cost = 0.4385 ± 0.6130, Reachability = 1.0826 ± 3.8216, Total Normalised = 1.2132
target_embedding: Cost = 0.4187 ± 0.5708, Reachability = 1.3905 ± 3.9968, Total Normalised = 1.2320
difference_vector: Cost = 0.1937 ± 0.3089, Reachability = 1.2900 ± 5.2814, Total Normalised = 0.6807
product_vector: Cost = 0.0141 ± 0.0209, Reachability = 0.0829 ± 0.1887, Total Normalised = 0.0000
cosine_similarity: Cost = 0.0738 ± 0.1030, Reachability = 0.3555 ± 0.6680, Total Normalised = 0.1989
euclidean_distance: Cost = 0.1062 ± 0.2469, Reachability = 0.1602 ± 0.4660, Total Normalised = 0.2336
dot_product: Cost = 0.0965 ± 0.1231, Reachability = 0.2528 ± 0.5312, Total Normalised = 0.2303
lemmatized: Cost = 0.1534 ± 0.2364, Reachability = 1.0887 ± 2.1529, Total Normalised = 0.5428
lexical_field_mask: Cost = 0.2669 ± 0.3855, Reachability = 2.0206 ± 3.4192, Total Normalised = 1.0089
pos_mask: Cost = 0.4185 ± 0.6652, Reachability = 4.7725 ± 7